# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
method = "Decision Tree"

print("Method chosen:", method)
print("Goal: rank pages for content review")
print("Metric: Precision@50")
print("Validation: client-level holdout")

Method chosen: Decision Tree
Goal: rank pages for content review
Metric: Precision@50
Validation: client-level holdout


Method: Decision Tree

I chose a Decision Tree because my goal is to rank pages that are more likely to need a content review. A decision tree can learn simple combinations of signals such as search visibility, position, traffic trend and content age.

It also fits this lane because the decisions are easy to understand and explain. I will compare it with my Week-4 hand-written baseline using the same data, split and metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Split the data by client

import pandas as pd
from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/ishigupgta1234-ux/flyrank-ml-internship/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Create the real target
df["label"] = (df["trend_direction"] == "down").astype(int)

# Get unique clients
clients = df["client_id"].unique()

# Keep 20% of clients for testing
train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

# Create train and test data
train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("Total rows:", len(df))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("Overlap in clients:",
      len(set(train_df["client_id"]) & set(test_df["client_id"])))

Total rows: 30000
Training rows: 26581
Test rows: 3419
Training clients: 25
Test clients: 7
Overlap in clients: 0


I will use a client-level holdout split.

Pages from the same client should not appear in both training and test data because that could make the model look better than it really is. I will keep about 20% of clients for testing and use the remaining clients for training.

This gives a more honest test of whether the model can rank pages for clients it has not seen during training.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X_train = train_df[features].fillna(0)
X_test = test_df[features].fillna(0)

y_train = train_df["label"]
y_test = test_df["label"]

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Model ranking
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

model_top50 = test_df.sort_values(
    "model_score", ascending=False
).head(50)

model_precision = model_top50["label"].mean()

# Week-4 baseline
test_df["stale"] = test_df["days_since_last_update"] >= 180
test_df["visible"] = test_df["impressions_90d"] >= 500

test_df["baseline_score"] = 0
test_df.loc[test_df["stale"], "baseline_score"] += 2
test_df.loc[test_df["visible"], "baseline_score"] += 1

baseline_top50 = test_df.sort_values(
    ["baseline_score", "days_since_last_update", "impressions_90d"],
    ascending=False
).head(50)

baseline_precision = baseline_top50["label"].mean()

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Decision Tree"],
    "Precision@50": [baseline_precision, model_precision]
})

print(comparison)

            Method  Precision@50
0  Week-4 Baseline          0.38
1    Decision Tree          0.68


In [12]:
print("Test-set base rate:", y_test.mean())
print("Majority-class base rate:", max(y_test.mean(), 1 - y_test.mean()))

Test-set base rate: 0.5238373793506873
Majority-class base rate: 0.5238373793506873


I will train a Decision Tree using page-level signals that are available before the outcome being predicted.

The target is whether the page is declining, based on trend_direction == "down". I will not use trend_direction or trend_pct as features because they are directly related to the target.

I will compare the model with my Week-4 hand-written baseline using Precision@50 on the same client-level test split.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Look at some model errors

test_df["prediction"] = model.predict(X_test)

errors = test_df[test_df["prediction"] != test_df["label"]]

print("Total errors:", len(errors))
print()
print(errors[
    ["content_id", "label", "prediction", "days_since_last_update",
     "impressions_90d", "avg_position"]
].head(10))

print()
print("Most important features:")

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance", ascending=False
)

print(importance.head(10))

Total errors: 1238

               content_id  label  prediction  days_since_last_update  \
13   content_a5a2fbc76336      0           1                     103   
15   content_689414059706      0           1                       8   
39   content_4595e8704e07      1           0                     104   
78   content_dea0d86223f3      0           1                      92   
100  content_cc97c093c379      0           1                      20   
179  content_552a9396d8dc      0           1                     104   
220  content_474bc8a4a3cb      1           0                      98   
222  content_cdeaa91ddaa5      0           1                      20   
229  content_5b377954b9b8      0           1                      20   
245  content_e79f03df2781      0           1                      20   

     impressions_90d  avg_position  
13               307          39.8  
15                38           7.8  
39                 4          36.3  
78                59           8.7  
10

The model can still make mistakes because a page being marked as declining does not always mean that it needs a content refresh.

I will look at some incorrect predictions and the most important features used by the model. This helps me understand what the model is actually using instead of only looking at the final score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.